# Workflow

This notebook contains the code for connecting all the specialized agents into one workflow.


**Needed Inputs**:
1. Topic
2. Level - beginner, intermediate or expert

**Specialized agents**:
1. Context Gathering Agent - curates external knowledge based on the topic
2. Planning Agent - Retrieves gathered context to structured plan based on the level
3. Teaching Agent - Teaches the student with indepth learning style (not spoon feeding)
4. Evaluation Agent - Scores, misconception and stores the information in memory
5. Final Report Agent - Gathers evaluation agent's reponse and combines it into structured insights

**Workflow Explanation**:
1. **Setup**: Orchestrator asks user for **Topic** + **Level** (beginner/intermediate/expert) → stores in long-term memory.
2. **Research Phase**:
   - Sequential agent runs:
     - Context Gathering Agent → collects information
     - Context Compaction Agent → summarizes & builds Knowledge Library → stores in long-term memory
   - Planning Agent → creates 5–10 milestones with subtopics using the knowledge library
   - Orchestrator shows syllabus to user & marks Research Phase complete.
3. **Action Phase** (starts when user says “let’s start”):
   - Teaching Agent uses memory (knowledge base, level, current milestone) → teaches concept → sends to Orchestrator → displayed to user.
4. **Doubts**: User asks doubt → Orchestrator → Teaching Agent → clarification → back to user.
5. **Evaluation**:
   - Evaluation Agent tests user understanding → records misconceptions, strengths/weaknesses, pass/fail for milestone → updates memory.
   - If passed → move to next milestone; repeat 3–5.
6. **Completion**:
   - After all milestones or user ends → Final Report Agent generates detailed learning journey report (strengths, weaknesses, recommendations) → sent to user via Orchestrator.

## Implementation

In [1]:
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool,ToolContext, AgentTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.memory import InMemoryMemoryService
from google.adk.tools import google_search,preload_memory,load_memory
from google.adk.code_executors import BuiltInCodeExecutor

from typing import Dict,Any
from dotenv import load_dotenv

In [2]:
# loading the environment
load_dotenv()

True

In [ ]:
SETUP_KEY_PREFIX = "setup:"
PHASE_KEY_PREFIX="phase:"
SYLLABUS_KEY_PREFIX = "syllabus:"
LEARNING_KEY_PREFIX = "learning:"
EVALUATION_KEY_PREFIX = "evaluation:"

### 1. Setup Phase

The setup phase uses `Agent` to interview the user and specialized **Function Tool** to capture the structured date, followed by an `after_agent_callback` to ensure the persistent in long term memory.

We use `learning:topic` key for storing the `topic` in the session, this `scope:key` is a convention designed for clarity and organization of session management like `user:`, `app:` or `temp:` called **descriptive prefixes**. 

In [ ]:
# custom tool to save the parameters to the session state (short term memory)
def set_learning_parameters(topic:str,level:str,tool_context:ToolContext)-> Dict[str,Any]:
    """
    Records the user's desired learning topic and expertise level.
    Args:
        topic - str : The subject matter the user wants to learn e.g. 'Linear Algebra'
        level - str : The expertise level e.g. 'beginner', 'intermediate', 'expert'
    """
    tool_context.state[f'{SETUP_KEY_PREFIX}topic'] = topic
    tool_context.state[f'{SETUP_KEY_PREFIX}level'] = level
    tool_context.state[f'{SETUP_KEY_PREFIX}complete'] = True

    return {"status":"success","message":"Learning Parameters saved to session state"}

setup_tool = FunctionTool(set_learning_parameters)

For storing the information in the long term memory we use `auto_save_preferences` custom function to transfer the session state to the memory service.

In [ ]:
async def auto_save_preferences(callback_context):
    """
    Automatically saves the session data (including parameters) to the long term memory after each agent turn
    """
    session_state = callback_context.state
    session_state[f"{PHASE_KEY_PREFIX}setup_done"] = True
    if session_state.get(f"{SETUP_KEY_PREFIX}complete") == True:
        await callback_context._invocation_context.memory_service.add_session_to_memory(
            callback_context._invocation_context.session
        )
        print("Setup parameters successfully persisted to long-term memory")
    else:
        print("Setup not marked as complete. Skipping memory save.")

We use `LlmAgent` instead of `Agent` because it's the descriptive name for the `Agent` wrapper. Here `LlmAgent` runs in reasoning loop (Think -> Act -> Observe) and it is the best practices for *production code* in multi-agent system and complex architecture.

In [5]:
setup_agent = LlmAgent(
    name="SetupAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""
    You are the setup orchestrator. Your first and only task is to greet the user and ask them two questions:
    1. What topic do they want to learn about ? and 
    2. What level are they (beginner, intermediate, expert)?

    Once you have both pieces of information, you MUST call the `set_learning_parameters` tool with the 'topic' and 'level' before responding to the user.
    """,
    tools=[setup_tool],
    after_agent_callback=auto_save_preferences
)

`Runner` is used for *production level* application

In [20]:
# the setup phase
session_service = InMemorySessionService()
memory_service = InMemoryMemoryService()

learning_runner = Runner(
    agent = setup_agent,
    app_name = "SkillixAI",
    session_service = session_service,
    memory_service= memory_service
)

In [33]:
# testing the setup phase
async def run_chat_loop():
    print(f"Setup is done")
    session_id = "test-session-001"
    user_id = "test-user-001"

    while True:
        user_text = input("\nYou: ")
        print("\nYou: ",user_text)
        if user_text.lower() in ['quit','exit']:
            break

        response = await learning_runner.run_debug(
            user_id=user_id,
            session_id=session_id,
            user_messages=user_text
        )

        print("\nAgent: ",response)

In [35]:
await run_chat_loop()

Setup is done

You:  i want to learn linear algebra

 ### Continue session: test-session-001

User > i want to learn linear algebra
SetupAgent > You mentioned you want to learn 'Linear Algebra'. Great choice!

What is your current level of expertise in Linear Algebra? Are you a 'beginner', 'intermediate', or 'expert'?
Setup not marked as complete. Skipping memory save.

Agent:  [Event(model_version='gemini-2.5-flash-lite', content=Content(
  parts=[
    Part(
      text="""You mentioned you want to learn 'Linear Algebra'. Great choice!

What is your current level of expertise in Linear Algebra? Are you a 'beginner', 'intermediate', or 'expert'?"""
    ),
  ],
  role='model'
), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=38,
  prompt_token_count=300,
  prompt_tokens_details=[
 

SetupAgent > All set! I've saved your preferences. You want to learn about Linear Algebra at a beginner level. Let's get started!
Setup parameters successfully persisted to long-term memory

Agent:  [Event(model_version='gemini-2.5-flash-lite', content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'level': 'beginner',
          'topic': 'Linear Algebra'
        },
        id='adk-eeed3bfe-a5da-49e5-8fc6-7a7df3753503',
        name='set_learning_parameters'
      )
    ),
  ],
  role='model'
), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=23,
  prompt_token_count=341,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=341
    ),
  ],
  total_token_count=364
), live_session_res

### 2. Research Phase

In this phase we will build specialized agents inside the `SequentialAgent` :
1. Context Gathering Agent
2. Context Compaction Agent
3. Planning Agent

#### Context Gathering Agent

This agent collects raw information using `google_search` tool

In [6]:
context_gathering_agent = LlmAgent(
    name="ContextGatheringAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    tools=[google_search],
    instruction="""You are a specialized research agent. Your task is to use the `google_search` tool to find the comprehensive, up-to-date information on the user's requested topic and level. The topic is retrieved from the Session State. Return the raw search results without summarizing them.""",
    output_key="raw_research"
)

#### Context Compaction Agent

This agent process the raw research into knowledge library. This step uses **Context Compaction** concept which consolidates raw events into a concise summary to improve performance and reduce future cost and stores the knowledge in the long term memory using `after_agent_callback`

In [7]:
async def persist_knowledge_to_memory(callback_context):
    """
    Callback to automatically save the compacted knowledge library to Long-term memory
    """

    memory_service = callback_context._invocation_context.memory_service
    current_session = callback_context._invocation_context.session

    await memory_service.add_session_to_memory(current_session)
    print("Knowledge Library successfully stored in Long-Term Memory")

In [8]:
context_compaction_agent = LlmAgent(
    name="ContextCompactionAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""Read the provided raw research findings : {raw_research}
    Your task is to act as a knowledge engineer. Create a detailed, concise and optimized 'Knowledge Library' summarizing the findings. This library must contain all key facts and concepts necessary to teach the user at thier specified level (from session state). Do not include conversation filler or irrelevant data.
    """,
    output_key="knowledge_library",
    after_agent_callback=persist_knowledge_to_memory
)

#### Planning Agent

Planning agent is to retrieve the necessary context-specifically the compacted **Knowledge Library** and the user's previously set **level** (from `preload_memory`) and use them to generate the learning milestones and stores the syllabus in the long term memory for evaluation and report agents

In [ ]:
# store the session state in long term memory
async def persist_milestones(callback_context):
    """
    Callback to automatically save the milestones (syllabus) to Long Term Memory after the Planning Agent completes its run.
    """
    # short term memory - current milestone index is initialized here
    callback_context.session.state[f"{LEARNING_KEY_PREFIX}current_milestone_index"] = 0
    callback_context.session.state[f"{PHASE_KEY_PREFIX}research_done"] = True

    # long term memory
    memory_service = callback_context._invocation_context.memory_service
    current_session = callback_context._invocation_context.session

    if current_session.state.get(f"{SYLLABUS_KEY_PREFIX}milestones"):
        await memory_service.add_session_to_memory(current_session)
        print("Syllabus milestones successfully persisted to Long Term Memory")
    else:
        print("Syllabus not found in Session State. Skipping memory save.")

In [ ]:
planning_agent = LlmAgent(
    name="PlanningAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""
    You are a curriculum planning specialist. Your task is to generate a comprehensive learning syllabus (5 to 10 milestone) based on the user's requested topic and expertise level, drawing heavily on the knowledge library provided from memory. User level (from Session State) : {learning:level} Topic (from Session State): {learning:topic} For each milestone, provide a clear title and 3 to 5 subtopics appropriate for the specified level. The output must be a clean, structured list
    """,
    tools=[preload_memory],
    output_key=f"{SYLLABUS_KEY_PREFIX}milestones",
    after_agent_callback=persist_milestones,
)

#### Research Agent

The `SequentialAgent` pipeline is used for the research phase

In [11]:
research_agent = SequentialAgent(
    name="ResearchAgent",
    sub_agents = [
        context_gathering_agent,
        context_compaction_agent,
        planning_agent
    ]
)

### 3. Action Phase

The teaching agent is the core agent for interacting with the user. This agent needs to access both the short term memory for conversation history and persistent long term memory. So the teaching agent can know current learning milestone, expertise level, strength and weakness of the user and keeping that in memory it would teach the concept. It also needs to know when to teach the topic and when to clear the doubts.

In [ ]:
teaching_agent = LlmAgent(
    name="TeachingAgent",
    tools=[preload_memory],
    instruction="""You are a highly personalized teaching agent. Your task is to educate the user based on thier specific learning.
    # Context Retrieval
    - **Topic** : {learning:topic}
    - **Expertise Level** : {learning:level}
    - **Current Milestone**: {learning:current_milestone_index}
    - **Syllabus**: {syllabus:milestones}
    - **Knowledge Library**: Content proactively loaded from MemoryService

    # Adaptive Teaching Profile
    The system has checked the user's historical performance (misconceptions, strengths/weakness) from Long Term Memory. 
    - **Previous Strengths for this Milestone**: [Content retireved from MemoryService, or 'None' if empty]
    - **Previous Weakness/Misconception**: [Content retrieved from MemoryService, or 'None' if empty]

    # Core Operating Procedure
    1. Analyze the user input - determine the user's intent by reviewing the current query and the session events
    2. Handling Clarifications (Doubt Phase) :
        - IF the user's input refers directly to the material you just presented (e.g., asking 'what does X mean?', 'can you rephrase?'), treat it as a **Doubt/Clarification Session**
        - Use the **Session Events** (short term memory) to review the immediate context and provide a coherent, specific clarification. Do not advance the lesson.
    
    3. Handling Instruction (Teaching Phase):
        - IF the user's input is a simple conversational opener or signals readiness, begin or continue the lesson for the current milesonte
        - Adaptive Delivery (the initial empty scenario):
            - IF previous weakness or misconception is 'None' (i.e., this is the first time teaching this milestone or the user passed successfully), proceed with a comprehensive, tailored introduction to the **Current Milestone** using **Knowledge Library**
            - ELSE (if weakness exists), start the lesson by explicitly addressing those weaknesses and correcting the misconceptions before introducing the new material. Use the 'Strengths' to reinforce succesful learning patterns.
    
    4. Final Output : Deliver the lessor or clarification, ensuring motivational tone and complexity match the user defined expertise level
    """
)

### 4. Evaluation Phase

In the evaluation phase, we will use the `EvaluationAgent` a single agent for understanding the user's learning progress and stores the persistent recording results in the long term memory. This agent decides whether the milestone is completed or not.

In [ ]:
# long term memory storage
async def persist_evaluation_results(callback_context):
    """
    Automatically saves the session contents (including structured evaluation results stored in session state during assessment) to long-term memory
    """
    await callback_context._invocation_context.memory_service.add_session_to_memory(
        callback_context._invocation_context.session
    )
    print("Evaluation results stored in long term memory")

We need to store the structured results to the session as intended so we are using a custom tool called `save_evaluation_results`. This tools would accept the assessment outcomes as structured arguments `pass_fail_status`, `misconceptions` and write them in the session state. The next function tool is `get_milestone_status` (refer the `retrieve_userinfo` in the 5 day course of the Agents for this part) to retrieve the current milestone index. Another function tool is `advance_milestone_index` to update the milestone index value.

In [ ]:
# short term memory storage

# tool to store structured evaluation results
def save_evaluation_results(
        tool_context: ToolContext,
        milestone_id: str,
        pass_fail_status:str,
        misconceptions: str,
        strengths: str,
        weakness: str,
) -> Dict[str,Any]:
    """
    Records detailed assessment results for a given milestone in the session state. This data is immediately available to other agent/tools in the session and is automatically transferred to Long-Term Memory via a persistnece callback attached to the Evaluation Agent.

    Args:
        - milestone_id - str : The unique identifier or index of the milestone being accessed
        - pass_fail_status - str : The outcome of the assessment (e.g., 'PASS' or 'FAIL')
        - misconceptions - str : A summary of the user's identified misconceptions
        - strengths - str : A summary of the user's strengths in the topic
        - weakness - str : A summary of the user's weaknesses in the topic
    """

    tool_context.state[f"{EVALUATION_KEY_PREFIX}milestone_id"] = milestone_id
    tool_context.state[f"{EVALUATION_KEY_PREFIX}status"] = pass_fail_status
    tool_context.state[f"{EVALUATION_KEY_PREFIX}misconceptions"]=misconceptions
    tool_context.state[f"{EVALUATION_KEY_PREFIX}strengths"]=strengths
    tool_context.state[f"{EVALUATION_KEY_PREFIX}weaknesses"]=weakness

    return {"status":"success","message":f"Evaluation results for {milestone_id} saved to Session State."}

In [13]:
def get_milestone_status(tool_context:ToolContext) -> Dict[str,Any]:
    """
    Retrieves the current milestone index and the full syllabus structure from the session state for evaluation context
    """
    current_index = tool_context.state.get(f"{LEARNING_KEY_PREFIX}current_milestone_index",0)
    syllabus_milestone = tool_context.state.get("syllabus:milestones","Syllabus not yet defined")

    return {
        "status": "success",
        "current_milestone_index":current_index,
        "syllabus_structure":syllabus_milestone
    }

In [14]:
def advance_milestone_index(tool_context: ToolContext) -> Dict[str,Any]:
    """
    Increments the current milestone index in the session state, moving the user to the next milestone in the learning journey
    """
    current_index = tool_context.state.get(f"{LEARNING_KEY_PREFIX}current_milestone_index",0)
    new_index = current_index + 1
    tool_context.state[f"{LEARNING_KEY_PREFIX}current_milestone_index"] = new_index

    return {
        "status":"success",
        "old_index":current_index,
        "new_index":new_index,
        "message":f"Milestone index advanced from {current_index} to {new_index}"
    }

`CalculationAgent` is a specialized agent to handle the objective logic and calculations reliably by generating python code executed in a sandbox, which avoids LLM guesswork (improves reliability)

In [7]:
calculation_agent = LlmAgent(
    name="CalculationAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""You are a specialized calculator that ONLY respond with Python code. Your task is to take a request for a calculation (e.g. checking facts or calculating scores) and translate it into a single block of python code that calculates the result.
    RULES:
    1. Your output MUST be ONLY a Python code block.
    2. Do NOT write any text before or after the code block
    3. The python code MUST print the final result to stdout
    4. You are PROHIBITED from performing the calculation yourself
    """,
    code_executor=BuiltInCodeExecutor(),
)

The `EvaluationAgent` will use the `CalculationAgent` as a `AgentTool` and assess the user's score

In [ ]:
calculation_tool = AgentTool(agent=calculation_agent)
save_results_tool = FunctionTool(save_evaluation_results)
get_status_tool = FunctionTool(get_milestone_status)
advance_index_tool = FunctionTool(advance_milestone_index)

evaluation_agent = LlmAgent(
    name="EvaluationAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""
    You are the Evaluation Agent - a precise assessor of the user's mastery of the current learning milestone.
Your sole responsibility is to:
1. Determine whether the user has sufficiently mastered the current milestone.
2. Produce a structured evaluation (pass/fail + detailed feedback) that will be automatically persisted to long-term memory.

### Strict Workflow (follow exactly, no exceptions):

1. **Gather Context**
   - Call `get_milestone_status()` to retrieve:
     • current_milestone_index
     • the full syllabus structure
   - Use this + conversation history to identify exactly what topic/content is being assessed right now.

2. **Conduct the Assessment**
   - Ask probing questions, present problems, or request explanations - whatever is most appropriate for the milestone.
   - You may spread the assessment over multiple turns if needed.

3. **Perform Any Required Calculations Using the Dedicated Tool**
   - For scoring, fact-checking, mathematical verification, or any objective computation:
        • Generate pure Python code that computes the result
        • Call the `CalculationAgent` tool (via the provided calculation_tool)
        • NEVER calculate scores or verify answers yourself
   - Example legitimate use: automatically grading multiple-choice answers, calculating percentage correct, checking numerical solutions, etc.

4. **When Assessment is Complete → Finalize and Record Results**
   After you are fully satisfied with the evaluation, make ONE final tool call to `save_evaluation_results` with the following arguments:
   - milestone_id: the identifier of the current milestone (usually the index or its title/id from the syllabus)
   - pass_fail_status: strictly "PASS" or "FAIL" (uppercase)
   - misconceptions: concise bullet-point or paragraph summary of specific misconceptions observed
   - strengths: concise summary of what the user demonstrated well
   - weakness: concise summary of gaps or errors (use "weaknesses" spelling as in the tool signature)

   This is the ONLY way evaluation data reaches long-term memory. The `persist_evaluation_results` callback triggers automatically after your final response ONLY if this tool has been called.

5. **If User Passed → Advance to Next Milestone**
   - Immediately after saving results with "PASS", call `advance_milestone_index()` so the curriculum progresses.

6. **Response Style**
   - Be encouraging but honest.
   - Clearly explain the reasoning behind pass/fail.
   - When the assessment is finished, end with a clear statement like:
     "Assessment complete. Recording results and preparing next steps…"

### Forbidden
- Do not attempt to save results in any way other than calling `save_evaluation_results`.
- Do not advance the milestone index on a "FAIL".
- Do not perform calculations manually — always delegate to CalculationAgent.
- Do not fabricate or guess evaluation keys — use only the tools provided.

Execute this protocol rigorously. Accurate, structured evaluation is critical for the student's long-term progress tracking.""",
    tools = [preload_memory,calculation_tool,save_results_tool,get_status_tool,advance_index_tool],
    after_agent_callback=persist_evaluation_results,
)

### 5. Report Phase

This phase contains only one specialized agent `FINALREPORTAGENT` and relies on memory for generating the report.

In [16]:
final_report_agent = LlmAgent(
    name="FinalReportAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""You are the Final Report Agent. Your primary task is to query the Long-Term Memory
        Service to synthesize a comprehensive detailed learning journey report for the user.

        You MUST follow these steps:
        1. Use the `load_memory` tool to retrieve ALL persistent learning records. This includes:
           - The original syllabus/milestones.
           - All recorded evaluation metrics (misconceptions, strengths, weakness, and pass/fail status for each milestone).
           - The compiled Knowledge Library.

        2. Synthesize the retrieved data into a detailed final report, which must include:
           - A summary of the completed syllabus and learning topics.
           - An overall assessment of the user's major learning **strengths** and **weaknesses** across all milestones.
           - Specific feedback regarding recurring **misconceptions**.
           - Clear, actionable **recommendations** for future learning or next steps.

        3. Present the final report clearly and professionally to the user. Do not perform any further actions.""",
        tools=[load_memory]
)

### Orchestrator

The `root_orchestrator_agent` connects all the phases and it is a high-level *workflow coordinator agent*. This agent uses the conditional logic in it's instructions to determine which specialized sub-agent should take control. 

In [ ]:
def get_workflow_status(tool_context: ToolContext) -> Dict[str, Any]:
    """Retrieves the current workflow status indicators from Session State."""
    research_done = tool_context.state.get(f"{PHASE_KEY_PREFIX}research_done", False)
    current_index = tool_context.state.get(f"{LEARNING_KEY_PREFIX}current_milestone_index", 0)
    total_milestones = tool_context.state.get("syllabus:total_milestones", 0)

    status = "Setup"
    if research_done and current_index < total_milestones:
        status = "Teaching"
    elif research_done and current_index >= total_milestones and total_milestones > 0:
        status = "Completion"

    return {
        "status": status,
        "research_done": research_done,
        "current_milestone_index": current_index,
        "total_milestones": total_milestones
    }
status_tool = FunctionTool(get_workflow_status)

In [ ]:
workflow_coordinator_agent = LlmAgent(
    name="WorkflowCoordinatorAgent",
    model=Gemini(model="gemini-2.5-flash-lite"),
    instruction="""You are the master orchestrator for the Personalized Learning Journey.
    Your primary goal is to manage the user's progression through phases (Setup, Research, Teaching, Evaluation, Completion).

    **CRITICAL STEP:** You MUST first call the `get_workflow_status` tool to determine the current phase status (Setup, Teaching, or Completion).

    Based on the status:
    1. **Status: Setup** (research_done=False): Call the `SetupAgent` tool to capture user preferences.
    2. **Status: Research** (After setup): Call the `ResearchSequentialAgent` tool.
    3. **Status: Teaching** (Ongoing): Call the `TeachingAgent` or `EvaluationAgent` tool based on the user's latest query (e.g., if the user asks to "take a quiz," call the Evaluation Agent or when a milestone topic is completed use the EvaluationAgent to test the user's understanding on the current milestone).
    4. **Status: Completion**: Call the `FinalReportAgent` tool to generate the final output.
    
    Always use the specialized agent tools for their designated tasks.""",
    tools=[
        status_tool,
        AgentTool(setup_agent),
        AgentTool(research_agent),
        AgentTool(teaching_agent),
        AgentTool(evaluation_agent),
        AgentTool(final_report_agent),
    ],
)